# Cliente HTTP OAI

Prueba aislada de la función HTTP compartida. La celda de la función se copia exactamente desde `nodes.py` y puede editarse antes de trasladar un cambio al pipeline.


In [ ]:
import inspect
import os
import time
import xml.etree.ElementTree as ET
from urllib.parse import urlencode

import certifi
import pandas as pd
import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning


In [ ]:
def get_oai_response(
    base_url,
    verify=None,
    max_retries=3,
    backoff_factor=1.0,
    min_interval=0.0,
    timeout=30.0,
):

    # Usa el bundle de certifi para evitar errores de certificado en requests
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    VERIFY_SSL = os.getenv("OAI_VERIFY_SSL", "false").lower() == "true"
    CA_BUNDLE = os.getenv("OAI_CA_BUNDLE") or certifi.where()
    requests.packages.urllib3.disable_warnings(category=InsecureRequestWarning)

    verify_param = CA_BUNDLE if VERIFY_SSL else False
    if verify is not None:
        verify_param = verify

    for attempt in range(1, max_retries + 1):
        start_time = time.time()
        response = None
        error = None
        try:
            response = requests.get(base_url, verify=verify_param, timeout=timeout)
        except requests.RequestException as exc:
            error = exc
        elapsed_time = time.time() - start_time

        if min_interval > 0:
            wait_time = max(min_interval - elapsed_time, 0)
            if wait_time > 0:
                print(f"Pausando {wait_time:.2f} segundos para no saturar el servidor")
                time.sleep(wait_time)

        if error:
            print(f"Error en request (intento {attempt}/{max_retries}): {error}")

        if response is not None and response.status_code == 200:
            return response

        status = response.status_code if response is not None else "sin respuesta"
        print(f"Error: {status} (intento {attempt}/{max_retries})")

        if attempt < max_retries:
            backoff = backoff_factor * attempt
            print(f"Reintentando en {backoff:.2f} segundos...")
            time.sleep(backoff)
    return None


In [ ]:
options = catalog.load("params:oai_extract_options").copy()
options["env"] = "dev"
if options.get("date_windows"):
    options["date_windows"] = options["date_windows"][:1]
options


In [ ]:
url = f"{options['base_url'].rstrip('/')}/{options['context']}?verb=Identify"
response = get_oai_response(url)
assert response is not None and response.ok
response.text[:1000]
